# Phase 7: Human review and persistence

The graph now saves each report and pauses until a person reviews it. Its state is stored in SQLite, so a paused report survives a restart.

One report is one *thread* (`thread_id` = report id). The saved state of a thread is the graph's short-term memory.

In [ ]:
import json

from community_voice.db.repository import list_pending, reset_database
from community_voice.graph.persistence import get_checkpointer
from community_voice.graph.workflow import build_graph
from community_voice.rag.reports import reset_reports
from community_voice.schemas.report import ReportIn

## 1. Start fresh

Clear the report database and the similarity index from earlier runs.

In [ ]:
reset_database()
reset_reports()

## 2. Submit a report

Live wires near a hospital: a critical report skips the web check, so it always reaches review. The graph runs, saves the report, and stops. What it prints is what the reviewer would see.

In [ ]:
graph = build_graph(get_checkpointer())
config = {"configurable": {"thread_id": "r05"}}
inputs = {
    "report_id": "r05",
    "report": ReportIn(
        text="After last night's storm there are live wires hanging low over the "
        "road near Tembisa Hospital. Children walk under them to get to school."
    ),
}
result = graph.invoke(inputs, config, version="v2")
print("Paused at:", graph.get_state(config).next)
print(json.dumps(result.interrupts[0].value, indent=2, ensure_ascii=False))

## 3. The review queue

Read from the database, most urgent first.

In [ ]:
for report in list_pending():
    print(report.id, report.urgency, report.status, report.department)

## 4. Restart the kernel

Click **Restart** in the toolbar. Everything in memory is gone, including the graph. The report is safe on disk.

Then run only the cells below. Skip sections 1 to 3.

In [ ]:
from langgraph.types import Command

from community_voice.db.repository import get_report, list_pending, list_reviews
from community_voice.graph.persistence import get_checkpointer
from community_voice.graph.workflow import build_graph

graph = build_graph(get_checkpointer())
waiting = list_pending()
print([report.id for report in waiting])

## 5. Resume with a correction

The reviewer approves, but sends it to the emergency line. The answer becomes the return value of `interrupt()`.

In [ ]:
report_id = waiting[0].id
config = {"configurable": {"thread_id": report_id}}
answer = {
    "action": "approve",
    "department": "Ekurhuleni Electricity department (emergency line)",
    "note": "Live wires near a hospital and a school: send the emergency crew.",
}
result = graph.invoke(Command(resume=answer), config, version="v2")
print("Status:", result.value["status"])

## 6. What is in the database

In [ ]:
print(get_report(report_id).model_dump_json(indent=2))
print(list_reviews(report_id))
print("Still waiting:", list_pending())

## What to notice

- The graph stopped and waited, then continued from the same place after a restart.
- The reviewer's correction was applied and the review was recorded.
- Code before `interrupt()` runs again on resume, so that step has no side effects. Saving happens in separate steps.
- A report that the web check finds to be a planned outage skips review, so it would not appear in the queue.